In [10]:
print("ok")

ok


In [11]:
%pwd

'c:\\Users\\91983\\OneDrive\\Desktop\\medical-chatbot-\\research'

In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 

In [13]:
# extract the data from the pdf file
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents



In [14]:
import os

os.chdir(r"C:\Users\91983\OneDrive\Desktop\medical-chatbot-")

print(os.getcwd())
print(os.path.exists("Data"))
print(os.listdir("Data"))

C:\Users\91983\OneDrive\Desktop\medical-chatbot-
True
['Medical_book.pdf']


In [15]:
extracted_data = load_pdf_file(data='Data/')

In [16]:
%pwd

'C:\\Users\\91983\\OneDrive\\Desktop\\medical-chatbot-'

In [17]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [18]:
text_chunks = text_split(extracted_data) 
print("length of text chunks:", len(text_chunks))

length of text chunks: 5860


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings 

In [3]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings 
# this model has 384 dimensions

In [4]:
embeddings = download_hugging_face_embeddings()
print("embedding model downloaded successfully")

c:\Users\91983\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


embedding model downloaded successfully


In [22]:
query_result=embeddings.embed_query("hello world")
print("query result:", query_result) 
print(len(query_result))

query result: [-0.034477267414331436, 0.031023245304822922, 0.006735002622008324, 0.02610897272825241, -0.03936199098825455, -0.16030247509479523, 0.06692401319742203, -0.006441466510295868, -0.047450482845306396, 0.014758829958736897, 0.07087530195713043, 0.05552758648991585, 0.01919335499405861, -0.026251377537846565, -0.010109559632837772, -0.02694047801196575, 0.022307416424155235, -0.02222663350403309, -0.1496925950050354, -0.01749303564429283, 0.0076762172393500805, 0.054352302104234695, 0.003254439914599061, 0.0317259281873703, -0.0846213549375534, -0.0294059868901968, 0.051595572382211685, 0.048124056309461594, -0.003314792178571224, -0.058279186487197876, 0.04196929186582565, 0.022210659459233284, 0.128188818693161, -0.022338882088661194, -0.011656327173113823, 0.06292837858200073, -0.03287631645798683, -0.0912260040640831, -0.031175334006547928, 0.052699558436870575, 0.04703483358025551, -0.08420311659574509, -0.0300561785697937, -0.020744871348142624, 0.009517832659184933, -

In [5]:
from dotenv import load_dotenv 
load_dotenv() 

True

In [10]:
import os 
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY 
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY 


In [9]:
import os 
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [25]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "medicalbot",
    "metric": "cosine",
    "host": "medicalbot-y1ehgf2.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [27]:
from langchain_pinecone import PineconeVectorStore 
dosearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
) 

In [11]:
#load the existing index
from langchain_pinecone import PineconeVectorStore 
dosearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name="medicalbot"
)

In [29]:
dosearch 

In [12]:
retriever=dosearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [31]:
retrieved_docs=retriever.invoke("wht is Acne")

In [32]:
retrieved_docs 

[Document(id='e00158e5-519a-4c2a-a320-c7ff2a91a740', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='403eb3d1-4b84-4dfc-a37a-94602913bcb3', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM 

In [17]:
# Use ChatOpenAI instead of OpenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature=0.4, max_tokens=500)

In [18]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [20]:
response = rag_chain.invoke({"input": "what is stats?"})
print(response["answer"])

I don't have enough information to answer the question as the context provided does not mention "stats."
